# EDA - creditcard.csv (Bank Credit Card Transactions)

## Objective
Exploratory data analysis of credit card fraud data:
- Data cleaning (missing values, duplicates, data types)
- Univariate and bivariate analysis
- Class imbalance quantification
- PCA feature correlation analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded')

## 1. Data Loading

In [ ]:
cc = pd.read_csv('../data/raw/creditcard.csv')
print('Shape:', cc.shape)
display(cc.head())
print(cc.dtypes)
print('Columns:', cc.columns.tolist())

## 2. Data Cleaning

In [ ]:
missing = cc.isnull().sum()
print('Missing:', 'None' if missing.sum()==0 else missing[missing>0])
print(f'Duplicates: {cc.duplicated().sum()}')
cc_clean = cc.drop_duplicates().copy()
print(f'After dedup: {len(cc_clean):,} (removed {len(cc)-len(cc_clean)})')
cc_clean['Class'] = cc_clean['Class'].astype('int8')
print(cc_clean.dtypes.head())

## 3. Class Imbalance

In [ ]:
cc_cnt = cc_clean['Class'].value_counts()
cc_pct = cc_clean['Class'].value_counts(normalize=True)*100
print(f'Legitimate (0): {cc_cnt[0]:,} ({cc_pct[0]:.2f}%)')
print(f'Fraudulent (1): {cc_cnt[1]:,} ({cc_pct[1]:.2f}%)')
print(f'Ratio: {cc_cnt[0]/cc_cnt[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cc_cnt.plot(kind='bar', ax=axes[0], color=['green','red'], edgecolor='black')
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_xticklabels(['Legitimate','Fraudulent'], rotation=0)
cc_pct.plot(kind='pie', ax=axes[1], labels=['Legitimate','Fraudulent'],
            autopct='%1.2f%%', colors=['green','red'])
axes[1].set_title('Class Distribution (%)', fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout()
plt.savefig('../data/processed/creditcard_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Univariate Analysis

In [ ]:
print(cc_clean.describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].hist(cc_clean['Time'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0,0].set_title('Time (seconds)', fontweight='bold')
axes[0,1].hist(cc_clean['Amount'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0,1].set_title('Amount', fontweight='bold')
axes[1,0].hist(np.log1p(cc_clean['Amount']), bins=50, edgecolor='black', alpha=0.7, color='lightgreen')
axes[1,0].set_title('Log(Amount)', fontweight='bold')
pca10 = [c for c in cc_clean.columns if c.startswith('V')][:10]
cc_clean[pca10].plot(kind='box', ax=axes[1,1])
axes[1,1].set_title('PCA Features V1-V10', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/creditcard_univariate.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cc_clean.boxplot(column='Amount', by='Class', ax=axes[0,0])
axes[0,0].set_title('Amount by Class', fontweight='bold')
cc_clean.boxplot(column='Time', by='Class', ax=axes[0,1])
axes[0,1].set_title('Time by Class', fontweight='bold')
la = cc_clean.loc[cc_clean['Class']==0,'Amount']
fa = cc_clean.loc[cc_clean['Class']==1,'Amount']
axes[1,0].hist(la, bins=50, alpha=0.6, label='Legitimate', color='green', edgecolor='black')
axes[1,0].hist(fa, bins=50, alpha=0.6, label='Fraudulent', color='red', edgecolor='black')
axes[1,0].set_title('Amount by Class', fontweight='bold'); axes[1,0].legend()
cr = cc_clean[['Time','Amount','Class']].corr()
sns.heatmap(cr, annot=True, fmt='.3f', cmap='coolwarm', center=0, ax=axes[1,1])
axes[1,1].set_title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/creditcard_bivariate.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Legit - Mean: ${la.mean():.2f}, Median: ${la.median():.2f}')
print(f'Fraud - Mean: ${fa.mean():.2f}, Median: ${fa.median():.2f}')

## 6. PCA Feature Analysis

In [ ]:
pca_cols = [c for c in cc_clean.columns if c.startswith('V')]
corr = cc_clean[[*pca_cols,'Class']].corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)
print('Top 15 PCA features by |corr| with Class:')
print(corr.head(15))

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
t15 = corr.head(15)
t15.plot(kind='barh', ax=axes[0], color=['green' if x>0 else 'red' for x in t15], edgecolor='black')
axes[0].set_title('Top 15 PCA Features Correlated with Fraud', fontweight='bold')
axes[1].hist(corr, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].set_title('Distribution of PCA Correlations', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/creditcard_pca_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
top6 = corr.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, feat in enumerate(top6):
    cc_clean.boxplot(column=feat, by='Class', ax=axes[i])
    axes[i].set_title(f'{feat} by Class', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/creditcard_top_features_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

In [ ]:
print(f'Records: {len(cc_clean):,}')
print(f'Legitimate: {cc_cnt[0]:,} ({cc_pct[0]:.2f}%)')
print(f'Fraudulent: {cc_cnt[1]:,} ({cc_pct[1]:.2f}%)')
print(f'Ratio: {cc_cnt[0]/cc_cnt[1]:.1f}:1')
print(f'Amount - Legit mean: ${la.mean():.2f}, Fraud mean: ${fa.mean():.2f}')
print('Top 5 PCA features:', corr.head(5).index.tolist())

report = f'''{'='*60}\nEDA SUMMARY - creditcard.csv\n{'='*60}\n"
Records: {len(cc_clean):,}\n"
Imbalance: {cc_pct[0]:.2f}% legit / {cc_pct[1]:.2f}% fraud\n"
{'='*60}'''
with open('../data/processed/Creditcard_EDA_Summary_Report.txt','w') as f: f.write(report)
print(report)

## 8. Save

In [ ]:
cc_clean.to_csv('../data/processed/creditcard_cleaned_eda.csv', index=False)
print(f'Saved creditcard_cleaned_eda.csv ({len(cc_clean):,} rows)')
print('Done!')